In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import sklearn
import pandas as pd
import os
import sys
import time
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F

print(sys.version_info)
for module in mpl, np, pd, sklearn, torch:
    print(module.__name__, module.__version__)
    
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print(device)


sys.version_info(major=3, minor=12, micro=3, releaselevel='final', serial=0)
matplotlib 3.10.1
numpy 2.2.4
pandas 2.2.3
sklearn 1.6.1
torch 2.7.0+cpu
cpu


In [2]:
from torchvision import datasets
from torchvision.transforms import ToTensor

# fashion_mnist图像分类数据集
train_ds = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

test_ds = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
)

# torchvision 数据集里没有提供训练集和验证集的划分
# 当然也可以用 torch.utils.data.Dataset 实现人为划分
# 从数据集到dataloader
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = torch.utils.data.DataLoader(test_ds, batch_size=16, shuffle=False)

In [3]:
from torchvision.transforms import Normalize

# 遍历train_ds得到每张图片，计算每个通道的均值和方差
def cal_mean_std(ds):
    mean = 0.
    std = 0.
    for img, _ in ds:
        mean += img.mean(dim=(1, 2))
        std += img.std(dim=(1, 2))
    mean /= len(ds)
    std /= len(ds)
    return mean, std


print(cal_mean_std(train_ds))
# 0.2860， 0.3205
transforms = nn.Sequential(
    Normalize([0.2860], [0.3205])
)


(tensor([0.2860]), tensor([0.3205]))


优点

自归一化：减少对 BatchNorm 的依赖。
   适合深层网络：尤其在全连接层中表现优异。
    梯度更稳定：负区非零梯度避免神经元死亡。

缺点

   仅适用于全连接层：在卷积层（CNN）或循环网络（RNN）中效果不如 ReLU。
   需要特定初始化：权重必须用 LeCun Normal 初始化（方差 1/n）。

In [4]:
class NeuralNetwork(nn.Module):
    def __init__(self, layers_num=2):
        super().__init__()
        self.transforms = transforms
        self.flatten = nn.Flatten()
        # 多加几层
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 100),  # in_features=784, out_features=300
            nn.ReLU(),
        )
        # 加19层
        for i in range(1, layers_num):
            self.linear_relu_stack.add_module(f"Linear_{i}", nn.Linear(100, 100))
            self.linear_relu_stack.add_module(f"selu", nn.SELU()) # 这里采用SELU激活函数
        # 输出层
        self.linear_relu_stack.add_module("Output Layer", nn.Linear(100, 10))
        
        # 初始化权重
        self.init_weights()
        
    def init_weights(self):
        """使用 xavier 均匀分布来初始化全连接层的权重 W"""
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        # x.shape [batch size, 1, 28, 28]
        x = self.transforms(x)
        x = self.flatten(x)  
        # 展平后 x.shape [batch size, 28 * 28]
        logits = self.linear_relu_stack(x)
        # logits.shape [batch size, 10]
        return logits

for idx, (key, value) in enumerate(NeuralNetwork(20).named_parameters()):
    print(f"Linear_{idx // 2:>02}\tparamerters num: {np.prod(value.shape)}")

Linear_00	paramerters num: 78400
Linear_00	paramerters num: 100
Linear_01	paramerters num: 10000
Linear_01	paramerters num: 100
Linear_02	paramerters num: 10000
Linear_02	paramerters num: 100
Linear_03	paramerters num: 10000
Linear_03	paramerters num: 100
Linear_04	paramerters num: 10000
Linear_04	paramerters num: 100
Linear_05	paramerters num: 10000
Linear_05	paramerters num: 100
Linear_06	paramerters num: 10000
Linear_06	paramerters num: 100
Linear_07	paramerters num: 10000
Linear_07	paramerters num: 100
Linear_08	paramerters num: 10000
Linear_08	paramerters num: 100
Linear_09	paramerters num: 10000
Linear_09	paramerters num: 100
Linear_10	paramerters num: 10000
Linear_10	paramerters num: 100
Linear_11	paramerters num: 10000
Linear_11	paramerters num: 100
Linear_12	paramerters num: 10000
Linear_12	paramerters num: 100
Linear_13	paramerters num: 10000
Linear_13	paramerters num: 100
Linear_14	paramerters num: 10000
Linear_14	paramerters num: 100
Linear_15	paramerters num: 10000
Linear_